<a href="https://colab.research.google.com/github/anumit2004/Attention-free-Transformer/blob/main/AFT_LOCAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AFT-local code base .

In [ ]:
import torch
import math
from torch import nn
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

## Mathematics and Theory of AFT-Local

**AFT-Local** is a variant of the Attention Free Transformer designed to emphasize local context and reduce the complexity of the global interaction. While the standard Transformer relies on $O(T^2)$ dot-product attention, AFT achieves linear or near-linear scaling by using an element-wise gating mechanism.

#### 1. The AFT-Full Foundation
In the base **AFT-Full** model, every token $t$ interacts with every other token $t'$ in the sequence:
$$Y_t = \sigma(Q_t) \odot \frac{\sum_{t'=1}^T \exp(w_{t,t'} + K_{t'}) \odot V_{t'}}{\sum_{t'=1}^T \exp(w_{t,t'} + K_{t'})}$$
where $w_{t,t'}$ is a learned pairwise position bias. While efficient, this still computes a full $T \times T$ interaction map.

#### 2. The AFT-Local Formulation
**AFT-Local** introduces a locality constraint via a sliding window of size $s$. Mathematically, this is achieved by modifying the position bias $w_{t,t'}$ such that interactions outside the window are nullified:

$$w_{t,t'} = \begin{cases} w_{t,t'} & \text{if } |t - t'| < s \\ -\infty & \text{otherwise} \end{cases}$$

When passed through the exponential function, $\exp(-\infty)$ becomes $0$, resulting in the gated output:
$$Y_t = \sigma(Q_t) \odot \frac{\sum_{t' \in \text{window}(t)} \exp(w_{t,t'} + K_{t'}) \odot V_{t'}}{\sum_{t' \in \text{window}(t)} \exp(w_{t,t'} + K_{t'})}$$

#### 3. Differentiation from AFT-Full
*   **Attention Span:** AFT-Full considers the entire sequence at once, which can lead to "diluted" focus for tasks where local patterns (like syntax or texture) are most important. AFT-Local acts similarly to a 1D Convolution with a large kernel but retains the dynamic gating of AFT.
*   **Inductive Bias:** By enforcing $|t - t'| < s$, the model is forced to learn features from the immediate neighborhood, which is often more effective for natural language and signal processing.

#### 4. Why AFT-Local is Better
1.  **Memory Scalability:** In optimized implementations, AFT-Local avoids the need to store or compute a full $T \times T$ position matrix, allowing it to handle much longer sequences than AFT-Full.
2.  **Focus & Signal-to-Noise:** By masking distant tokens, it prevents the model from being influenced by irrelevant information at the start of a long document when processing the end.
3.  **Efficiency:** It provides a middle ground between the global scope of Transformers and the efficiency of RNNs, making it highly suitable for high-resolution tasks.

###The `K_stable` and `w_stable` lines I included are standard deep learning computational trick for numerical stability, rather than a part of the theoretical AFT math.

In [ ]:
# =============================================================================
# AFT-LOCAL MODULE
# =============================================================================
class AFTLocal(nn.Module):
    def __init__(self, seq_len, dim,hidden_dim ,  window_size=16):
        super().__init__()
        self.dim = dim
        self.seq_len = seq_len
        self.window_size = window_size

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)

        # Factorized position bias to save parameters
        self.u = nn.Parameter(torch.randn(seq_len, hidden_dim) * 0.01)
        self.v = nn.Parameter(torch.randn(seq_len, hidden_dim) * 0.01)

        self.out_proj = nn.Linear(dim, dim)

        # Precompute the boolean mask for the local window
        self.register_buffer('local_mask', self._create_local_mask(seq_len, window_size))

    def _create_local_mask(self, seq_len, window_size):
        """Creates a boolean mask where True means |t - t'| < window_size."""
        positions = torch.arange(seq_len)
        diff = torch.abs(positions.unsqueeze(1) - positions.unsqueeze(0))
        # 1. Local window: |t - t'| < window_size
        local = diff < window_size

        # 2. Causal: t can only attend to t' where t' <= t
        causal = positions.unsqueeze(1) >= positions.unsqueeze(0)

        return local & causal

    def forward(self, x):
        B, T, d = x.shape

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q_sig = torch.sigmoid(Q)

        # Compute factorized position bias
        u = self.u[:T, :]
        v = self.v[:T, :]
        w = torch.matmul(u, v.transpose(0, 1))  # Shape: (T, T)

        # FIX: Mask distant entries with -inf so exp(-inf) becomes 0
        mask = self.local_mask[:T, :T]
        causal = torch.tril(torch.ones(T, T, device=mask.device, dtype=torch.bool))
        mask = mask & causal
        w = w.masked_fill(~mask, float('-inf'))

        # Numerical stability against exponential overflow
        K_stable = K - K.max(dim=1, keepdim=True)[0]

        # exp_K = torch.exp(K_stable)
        # exp_w = torch.exp(w) # Out-of-window values are now exactly 0

        # FIX: Stabilize w to prevent torch.exp(w) from overflowing to inf
        # We take the max along dim=1. The -inf masked values will be safely ignored.
        w_max = w.max(dim=1, keepdim=True)[0]
        w_stable = w - w_max

        exp_K = torch.exp(K_stable)
        exp_w = torch.exp(w_stable)

        # Matrix multiplication approach to avoid O(B * T * T * d) memory explosion
        numerator = exp_w @ (exp_K * V)
        denominator = exp_w @ exp_K + 1e-6

        # Calculate final context and gate it with Query
        context = numerator / denominator
        Y = Q_sig * context

        return self.out_proj(Y)

In [ ]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim, dp=0.1):
        super().__init__()
        self.l1 = nn.Linear(dim, hidden_dim)
        self.g1 = nn.GELU()
        self.l2 = nn.Linear(hidden_dim, dim)
        self.d1 = nn.Dropout(dp)

    def forward(self, x):
        x = self.l1(x)
        x = self.g1(x)
        x = self.d1(x)
        return self.l2(x)

### AFT Local Encoder Architecture

An AFT Encoder Block combines the AFT mechanism with a feed-forward network. In this implementation, we utilize **AFT-Local**, which replaces standard global attention with a windowed, attention-free mechanism for improved efficiency and local feature extraction.

1.  **Layer Normalization (ln1):** The input `x` is first normalized before being fed into the AFT layer.
  $$x_{norm1} = \text{LayerNorm}(x)$$

2.  **AFT-Local Attention (attn):** The normalized input passes through the `AFTLocal` mechanism. Unlike the full version, AFT-Local applies a **sliding window mask** of size $s$, ensuring that each token only interacts with its immediate neighbors.
  $$attn\_output = \text{AFTLocal}(x_{norm1})$$

3.  **Residual Connection and Dropout (d1):** A residual connection is added, and dropout is applied to the AFT output before being added back to the original input.
  $$x = x + \text{Dropout}(attn\_output)$$

4.  **Layer Normalization (ln2):** The result is normalized again before entering the MLP.
  $$x_{norm2} = \text{LayerNorm}(x)$$

5.  **Multi-Layer Perceptron (mlp):** A Feed-Forward Network (MLP) processes the normalized output using non-linear activations to learn complex transformations.
  $$mlp\_output = \text{MLP}(x_{norm2})$$

6.  **Residual Connection and Dropout (d2):** A final residual connection and dropout ensure stable gradient flow and prevent overfitting.
    $$out = x + \text{Dropout}(mlp\_output)$$

By using AFT-Local, this architecture gains a strong inductive bias for sequential data where local context is paramount, while maintaining the linear scaling benefits of the Attention Free Transformer.

In [ ]:
class AFTEncoderBlock(nn.Module):
    def __init__(self, max_seqlen, dim, hidden_dim,window_size ,p=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)

        # SWAPPED: Now using AFTLocal with a defined window_size (e.g., 16)
        self.attn = AFTLocal(seq_len=max_seqlen, dim=dim, hidden_dim=hidden_dim , window_size=window_size)

        self.mlp = MLP(dim, hidden_dim, dp=p)
        self.d1 = nn.Dropout(p)
        self.d2 = nn.Dropout(p)

    def forward(self, x):
        x_norm = self.ln1(x)
        x = x + self.d1(self.attn(x_norm))
        x_norm = self.ln2(x)
        out = x + self.d2(self.mlp(x_norm))
        return out

### AFT Model Architecture

The `AFT` class defines the complete Attention Free Transformer model. It takes an input sequence, processes it through multiple AFT Encoder Blocks, and then projects the output back to the vocabulary size for tasks like language modeling.

**Initialization (`__init__`):**

1.  **`vocab_size`**: The size of the input vocabulary.
2.  **`max_seqlen`**: The maximum sequence length the model can handle.
3.  **`dim`**: The embedding dimension for tokens and positional embeddings.
4.  **`hidden_dim`**: The hidden dimension used within the AFTFull and MLP components.
5.  **`depth`**: The number of AFT Encoder Blocks to stack (default is 4).
6.  **`p`**: Dropout probability.

*   **`self.embed`**: An `nn.Embedding` layer to convert input token IDs into dense vectors of size `dim`.
*   **`self.pos_embed`**: A learnable `nn.Embedding` layer for absolute positional embeddings. It assigns a unique embedding vector to each position up to `max_seqlen`.
*   **`self.enc`**: This is an `nn.Sequential` container that stacks `depth` number of `AFTEncoderBlock` instances. Each `AFTEncoderBlock` processes the sequence, applying AFT and MLP operations.
*   **`self.dec`**: An `nn.Linear` layer that acts as the decoder head, projecting the final `dim` sized output of the encoder stack back to `vocab_size`. This is typically used for predicting the next token in a sequence.

**Forward Pass (`forward`):**

Given an input `x` of shape `(Batch_size, Sequence_length)`:

1.  **Positional Encoding:**
    *   `positions = torch.arange(0, T, device=device).unsqueeze(0).expand(B, T)`: Creates a tensor of position indices for each element in the batch and sequence.
    *   `x = self.embed(x) * math.sqrt(self.dim) + self.pos_embed(positions)`: The input token embeddings are combined with positional embeddings. The token embeddings are scaled by `sqrt(self.dim)` as is common in Transformer architectures, and then summed with the learned positional embeddings.

2.  **Encoder Stack:**
    *   `x = self.enc(x)`: The combined embeddings pass through the stack of `AFTEncoderBlock`s. Each block processes the sequence, applying its AFT and MLP components, and incorporating residual connections and layer normalization.

3.  **Decoder Output:**
    *   `out = self.dec(x)`: The final output from the encoder stack is passed through the linear decoder layer, producing logits for each token in the vocabulary at each position.

The `AFT` model, therefore, provides a complete sequence-to-sequence architecture using the AFT mechanism as its primary attention-like component.

In [ ]:
class AFT(nn.Module):
    def __init__(self, vocab_size, max_seqlen, dim, hidden_dim,window_size, depth=4, p=0.1):
        super().__init__()
        self.dim = dim
        self.embed = nn.Embedding(vocab_size, dim)

        # Simple learnable absolute positional embeddings
        self.pos_embed = nn.Embedding(max_seqlen, dim)

        self.enc = nn.Sequential(*[
            AFTEncoderBlock(max_seqlen, dim, hidden_dim, window_size, p=p)
            for _ in range(depth)
        ])
        self.dec = nn.Linear(dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        device = x.device

        positions = torch.arange(0, T, device=device).unsqueeze(0).expand(B, T)
        x = self.embed(x) * math.sqrt(self.dim) + self.pos_embed(positions)

        x = self.enc(x)
        out = self.dec(x)
        return out


##Preparing the dataset  -  `OPUS`

In [ ]:
class OpusTextDataset(Dataset):
    """A clean dataset class that just holds pre-processed token chunks."""
    def __init__(self, input_ids_list):
        self.input_ids = input_ids_list

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        chunk = torch.tensor(self.input_ids[idx], dtype=torch.long)
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
def prepare_dataloaders(max_seqlen=64, batch_size=16, max_samples=5000, tokenizer_name="gpt2"):
    print("Loading OPUS Books dataset via Hugging Face...")
    raw_dataset = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")

    print("Initializing Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Tokenizing and chunking text corpus...")
    buffer = []
    all_input_ids = []

    for item in raw_dataset:
        text = item['translation']['en']
        tokens = tokenizer.encode(text)
        buffer.extend(tokens)

        while len(buffer) >= (max_seqlen + 1):
            all_input_ids.append(buffer[:max_seqlen + 1])
            buffer = buffer[max_seqlen:]


        if max_samples != None :
            if len(all_input_ids) >= max_samples:
                all_input_ids = all_input_ids[:max_samples]
                break

    # 1. Wrap all chunks in our dataset class
    full_dataset = OpusTextDataset(all_input_ids)

    # 2. Calculate split sizes (80% Train, 10% Validation, 10% Test)
    total_size = len(full_dataset)
    train_size = int(0.8 * total_size)
    val_size = int(0.1 * total_size)
    test_size = total_size - train_size - val_size

    # 3. Perform the random split
    generator = torch.Generator().manual_seed(42) # Seed for reproducibility
    train_data, val_data, test_data = random_split(
        full_dataset,
        [train_size, val_size, test_size],
        generator=generator
    )

    print(f"Data Split Complete: {len(train_data)} Train | {len(val_data)} Val | {len(test_data)} Test")

    # 4. Create DataLoaders
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, tokenizer

##Training the dataset - `OPUS`

In [ ]:
# =============================================================================
# 3. TRAINING ROUTINE WITH VALIDATION
# =============================================================================

def train_on_opus():
    # Hyperparameters
    MAX_SEQLEN = 64
    EMBED_DIM = 512
    HIDDEN_DIM = 512
    DEPTH = 15
    BATCH_SIZE = 16
    EPOCHS = 20
    LR = 8e-3
    WINDOW_SIZE = 32
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Fetch our newly split dataloaders
    train_loader, val_loader, test_loader, tokenizer = prepare_dataloaders(
        max_seqlen=MAX_SEQLEN,
        batch_size=BATCH_SIZE,
        max_samples=None
    )
    VOCAB_SIZE = tokenizer.vocab_size

    print(f"\nConfiguration Details:")
    print(f"-> Device: {DEVICE}")
    print(f"-> Model Vocabulary Size: {VOCAB_SIZE}")
    print("Initializing AFT Model Architecture...")

    # Build Model
    model = AFT(
        vocab_size=VOCAB_SIZE,
        max_seqlen=MAX_SEQLEN,
        dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        window_size = WINDOW_SIZE,
        depth=DEPTH
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()

    print("\nBeginning Training Pipeline...")

    for epoch in range(EPOCHS):
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 20 == 0:
                print(f"Epoch {epoch+1}/{EPOCHS} | Train Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

        avg_train_loss = train_loss / len(train_loader)
        train_perplexity = math.exp(avg_train_loss)
        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        correct_tokens = 0
        total_tokens = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)

                loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
                val_loss += loss.item()

                preds = outputs.argmax(dim=-1)
                correct_tokens += (preds == targets).sum().item()
                total_tokens += targets.numel()

        avg_val_loss = val_loss / len(val_loader)
        val_perplexity = math.exp(avg_val_loss)
        val_accuracy = (correct_tokens / total_tokens) * 100

        print(f"=== Epoch {epoch+1:02d} Complete ===")
        print(f"Train Loss: {avg_train_loss:.4f} | Train PPL: {train_perplexity:.2f} | Val Loss: {avg_val_loss:.4f} | Val PPL: {val_perplexity:.2f} | Val Accuracy: {val_accuracy:.2f}%\n")
    # Return the test_loader as well so you can run final evaluations later
    return model, tokenizer, test_loader

if __name__ == '__main__':
    model, tokenizer, test_loader = train_on_opus()

## 4. Testing the Model: Text Generation

In [ ]:

# =============================================================================
# FINAL TEST SET EVALUATION
# =============================================================================

def evaluate_test_set(model, test_loader, tokenizer, device):
    print("\n--- Running Final Evaluation on Unseen Test Set ---")
    model.eval()
    criterion = nn.CrossEntropyLoss()
    VOCAB_SIZE = tokenizer.vocab_size

    test_loss = 0.0
    correct_tokens = 0
    total_tokens = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(test_loader):
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass
            outputs = model(inputs)

            # Calculate loss
            loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
            test_loss += loss.item()

            # Calculate accuracy
            preds = outputs.argmax(dim=-1)
            correct_tokens += (preds == targets).sum().item()
            total_tokens += targets.numel()

    avg_test_loss = test_loss / len(test_loader)
    test_perplexity = math.exp(avg_test_loss)
    test_accuracy = (correct_tokens / total_tokens) * 100

    print(f"Final Test Loss: {avg_test_loss:.4f}")
    print(f"Final Test Perplexity : {test_perplexity :.4f}")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print("---------------------------------------------------\n")

# Set the device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Run the evaluation using the variables returned from train_on_opus()
evaluate_test_set(model, test_loader, tokenizer, DEVICE)

##Generating the text

In [ ]:

def generate_text(model, tokenizer, prompt, max_new_tokens=100, device='cpu', temperature=0.4):
    model.eval()

    # Safety check: Ensure the model knows its max_seqlen (since it wasn't in AFT __init__)
    if not hasattr(model, 'max_seqlen'):
        model.max_seqlen = 64

    encoded_prompt = tokenizer.encode(prompt, return_tensors='pt').to(device)
    generated_sequence = encoded_prompt.tolist()[0]

    print(f"\nGenerating text with prompt: '{prompt}'")

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Take only the last MAX_SEQLEN tokens if the sequence exceeds it
            current_input = torch.tensor([generated_sequence[-model.max_seqlen:]], dtype=torch.long).to(device)

            # Get predictions for the next token
            outputs = model(current_input)
            next_token_logits = outputs[0, -1, :] / temperature # Applying temperature

            # Sample the next token using multinomial distribution with temperature
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

            generated_sequence.append(next_token)

            # Stop if EOS token is generated
            if next_token == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated_sequence)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Testing
prompt = "The quick brown fox"
generated_text = generate_text(model, tokenizer, prompt, max_new_tokens=50, device=DEVICE)
print("\nGenerated Text 1:")
print(generated_text)

prompt_2 = "Once upon a time, in a land far, far away"
generated_text_2 = generate_text(model, tokenizer, prompt_2, max_new_tokens=50, device=DEVICE)
print("\nGenerated Text 2:")
print(generated_text_2)